# CSP Screener — Interactive Notebook

Two-step workflow:
1. **Quick Preview** (fast) — fetches stock-level data for all watchlist tickers in parallel
2. **Deep Scan** — select specific symbols, then scan their full option chains with your filters

## 1. Setup

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
import logging
import warnings

import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output
import ipywidgets as widgets

warnings.filterwarnings('ignore')
logging.getLogger('yfinance').setLevel(logging.ERROR)

import sys
sys.path.insert(0, '../')
from app.analysis.option_greeks import (
    black_scholes_put_delta,
    calc_otm_pct,
    calc_ann_roc,
    calc_rating,
    is_strong_fundamentals,
)
from app.config import settings

WATCHLIST = [
    'ADBE', 'AMZN', 'AVGO', 'BRK-B', 'GOOG',
    'META', 'MSFT', 'NFLX', 'NVDA', 'ORCL',
    'PEP', 'PLTR', 'PYPL', 'SAP', 'TSLA',
    'TSM', 'U', 'UNH', 'V',
]

DELAY = settings.yfinance_delay_between_symbols

print(f'Setup complete — {len(WATCHLIST)} tickers loaded')

## 2. Quick Preview

Fetches stock-level data (price, beta, PE, margins) in parallel — **no option chains**. Use this to pick which symbols to deep-scan.

In [ ]:
def fetch_quick_info(symbol):
    try:
        info = yf.Ticker(symbol).info or {}
        price = info.get('currentPrice') or info.get('regularMarketPrice')
        if not price:
            return {'symbol': symbol, 'error': 'no price'}
        beta = info.get('beta')
        pe = info.get('trailingPE')
        pm = info.get('profitMargins')
        pm_pct = round(pm * 100, 1) if pm is not None else None
        high = info.get('fiftyTwoWeekHigh')
        low = info.get('fiftyTwoWeekLow')
        range_pct = round((price - low) / (high - low) * 100, 1) if high and low and high != low else None
        strong = is_strong_fundamentals(pe, pm_pct, beta)
        return {
            'symbol': symbol, 'price': round(price, 2),
            'beta': round(beta, 2) if beta else None,
            'pe': round(pe, 1) if pe else None,
            'margin%': pm_pct,
            '52w_range%': range_pct,
            'strong': strong,
        }
    except Exception as e:
        return {'symbol': symbol, 'error': str(e)}


print(f'Fetching info for {len(WATCHLIST)} tickers in parallel...')
preview_rows = []
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(fetch_quick_info, s): s for s in WATCHLIST}
    for f in as_completed(futures):
        row = f.result()
        if 'error' in row:
            print(f"  {row['symbol']}: {row['error']}")
        else:
            preview_rows.append(row)

preview_df = pd.DataFrame(preview_rows).sort_values('symbol')
print(f'\nGot data for {len(preview_df)}/{len(WATCHLIST)} tickers:\n')
display(preview_df.style.format({'price': '${:.2f}', '52w_range%': '{:.0f}%'}))

## 3. Deep Scan

Select symbols from the preview above, adjust filters, then click **Run Scan** to fetch option chains for only the selected tickers.

In [ ]:
# --- Auto-select strong fundamentals from preview ---
strong_symbols = preview_df[preview_df['strong'] == True]['symbol'].tolist() if not preview_df.empty else []

symbol_selector = widgets.SelectMultiple(
    options=WATCHLIST,
    value=tuple(strong_symbols),
    description='Symbols:',
    layout=widgets.Layout(width='180px', height='220px'),
    style={'description_width': 'initial'},
)

select_hint = widgets.HTML(f'<i>Ctrl/Cmd+click to multi-select. <b>{len(strong_symbols)}</b> strong-fundamental tickers pre-selected.</i>')

# --- Pre-scan filter widgets ---
min_iv_w = widgets.FloatSlider(value=0.15, min=0.05, max=0.80, step=0.05, description='Min IV:', readout_format='.2f')
min_delta_w = widgets.FloatSlider(value=0.10, min=0.05, max=0.50, step=0.05, description='Min Delta:', readout_format='.2f')
max_delta_w = widgets.FloatSlider(value=0.35, min=0.05, max=0.99, step=0.05, description='Max Delta:', readout_format='.2f')
min_dte_w = widgets.IntSlider(value=14, min=7, max=90, step=7, description='Min DTE:')
max_dte_w = widgets.IntSlider(value=60, min=7, max=90, step=7, description='Max DTE:')
min_otm_w = widgets.FloatSlider(value=-20.0, min=-50.0, max=20.0, step=1.0, description='Min OTM%:', readout_format='.1f')
min_roc_w = widgets.FloatSlider(value=3.0, min=0.0, max=40.0, step=1.0, description='Min ROC%:', readout_format='.1f')
max_beta_w = widgets.FloatSlider(value=2.5, min=0.5, max=4.0, step=0.1, description='Max Beta:', readout_format='.1f')
max_capital_w = widgets.FloatSlider(value=75000, min=10000, max=200000, step=5000, description='Max Capital:', readout_format=',.0f')

filter_panel = widgets.VBox([
    widgets.HTML('<b>Pre-Scan Filters</b>'),
    min_iv_w,
    widgets.HBox([min_delta_w, max_delta_w]),
    widgets.HBox([min_dte_w, max_dte_w]),
    min_otm_w, min_roc_w, max_beta_w, max_capital_w,
])

# --- Scan button + output ---
scan_btn = widgets.Button(
    description='Run Scan',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px'),
)
scan_output = widgets.Output()

# --- Post-scan filter/sort controls ---
filter_sym_w = widgets.SelectMultiple(
    options=[], value=(), description='Symbol:',
    layout=widgets.Layout(width='150px', height='130px'),
    style={'description_width': 'initial'},
)
filter_rating_w = widgets.SelectMultiple(
    options=[('5 — Excellent', 5), ('4 — Good', 4), ('3 — Fair', 3), ('2 — Weak', 2), ('1 — Poor', 1)],
    value=(3, 4, 5),
    description='Rating:',
    layout=widgets.Layout(width='180px', height='130px'),
    style={'description_width': 'initial'},
)
filter_strong_w = widgets.Checkbox(value=False, description='Strong fundamentals only', indent=False)
sort_by_w = widgets.Dropdown(
    options=[
        ('Annualized ROC', 'ann_roc_pct'),
        ('IV', 'iv'),
        ('Delta', 'delta'),
        ('OTM %', 'otm_pct'),
        ('DTE', 'dte'),
        ('Bid', 'bid'),
        ('Capital Required', 'capital_required'),
    ],
    value='ann_roc_pct', description='Sort by:',
    style={'description_width': 'initial'},
)
sort_order_w = widgets.Dropdown(
    options=[('Descending', False), ('Ascending', True)],
    value=False, description='Order:',
    style={'description_width': 'initial'},
)
refine_btn = widgets.Button(
    description='Apply Filters',
    button_style='info',
    layout=widgets.Layout(width='200px', height='35px'),
)
refine_output = widgets.Output()

refine_panel = widgets.VBox([
    widgets.HTML('<b>Filter & Sort Results</b> <i>(works on already-scanned data, no re-scan)</i>'),
    widgets.HBox([
        filter_sym_w,
        filter_rating_w,
        widgets.VBox([filter_strong_w, sort_by_w, sort_order_w]),
    ]),
    refine_btn,
    refine_output,
])


# --- Shared styled display helper ---
def _color_rating(val):
    colors = {
        5: 'background-color: #d4edda; color: #155724',
        4: 'background-color: #c3e6cb; color: #155724',
        3: 'background-color: #fff3cd; color: #856404',
        2: 'background-color: #f8d7da; color: #721c24',
        1: 'background-color: #f5c6cb; color: #721c24',
    }
    return colors.get(val, '')


def _styled_table(df):
    styled = df.style.map(_color_rating, subset=['rating'])
    styled = styled.format({
        'price': '${:.2f}', 'strike': '${:.2f}', 'bid': '${:.2f}', 'mid': '${:.2f}',
        'capital_required': '${:,.0f}', 'otm_pct': '{:.1f}%',
        'ann_roc_pct': '{:.1f}%', 'iv': '{:.1%}', 'delta': '{:.3f}',
        'pe_ratio': '{:.1f}', 'beta': '{:.2f}', 'profit_margin': '{:.1f}%',
    })
    return styled


# --- Scan functions ---
def get_filters():
    return {
        'min_iv': min_iv_w.value,
        'min_delta': min_delta_w.value,
        'max_delta': max_delta_w.value,
        'min_dte': min_dte_w.value,
        'max_dte': max_dte_w.value,
        'min_otm_pct': min_otm_w.value,
        'min_ann_roc': min_roc_w.value,
        'max_beta': max_beta_w.value,
        'max_capital': max_capital_w.value,
    }


def _fetch_and_screen(symbol, filters, cached_info=None):
    ticker = yf.Ticker(symbol)
    if cached_info:
        price = cached_info.get('price')
        pe_ratio = cached_info.get('pe')
        beta = cached_info.get('beta')
        pm = cached_info.get('margin%')
    else:
        info = ticker.info or {}
        price = info.get('currentPrice') or info.get('regularMarketPrice')
        if not price:
            raise ValueError(f'No price for {symbol}')
        pe_ratio = info.get('trailingPE')
        beta = info.get('beta')
        pm = info.get('profitMargins')
        if pm is not None:
            pm = pm * 100

    strong = is_strong_fundamentals(pe_ratio, pm, beta)
    if beta is not None and beta > filters['max_beta']:
        return []

    expirations = ticker.options
    if not expirations:
        raise ValueError(f'No options for {symbol}')

    today = date.today()
    valid_exps = []
    for exp_str in expirations:
        try:
            dte = (date.fromisoformat(exp_str) - today).days
            if filters['min_dte'] <= dte <= filters['max_dte']:
                valid_exps.append((exp_str, dte))
        except ValueError:
            continue
    valid_exps = valid_exps[:5]
    if not valid_exps:
        return []

    results = []
    for i, (exp_str, dte) in enumerate(valid_exps):
        if i > 0:
            time.sleep(DELAY)
        try:
            chain = ticker.option_chain(exp_str)
        except Exception:
            continue
        if chain is None or chain.puts is None or chain.puts.empty:
            continue

        for _, put in chain.puts.iterrows():
            strike = put.get('strike')
            bid = put.get('bid')
            ask = put.get('ask')
            iv = put.get('impliedVolatility')
            if strike is None or bid is None or iv is None:
                continue
            if pd.isna(bid) or pd.isna(iv) or bid <= 0:
                continue

            mid = (bid + (ask if pd.notna(ask) else bid)) / 2
            otm_pct = calc_otm_pct(price, strike)
            ann_roc = calc_ann_roc(bid, strike, dte)
            capital = strike * 100
            delta_abs = abs(black_scholes_put_delta(price, strike, dte / 365, 0.05, iv))
            rating, label = calc_rating(iv, delta_abs, dte, ann_roc, strong)

            if iv < filters['min_iv']:
                continue
            if not (filters['min_delta'] <= delta_abs <= filters['max_delta']):
                continue
            if otm_pct < filters['min_otm_pct']:
                continue
            if ann_roc < filters['min_ann_roc']:
                continue
            if capital > filters['max_capital']:
                continue

            results.append({
                'symbol': symbol, 'price': price, 'strike': strike,
                'expiry': exp_str, 'dte': dte,
                'bid': round(bid, 2), 'mid': round(mid, 2),
                'iv': round(iv, 4), 'delta': round(delta_abs, 4),
                'otm_pct': round(otm_pct, 2), 'ann_roc_pct': round(ann_roc, 2),
                'capital_required': capital,
                'pe_ratio': pe_ratio, 'beta': beta,
                'profit_margin': pm,
                'strong_fundamentals': strong,
                'rating': rating, 'rating_label': label,
            })

    results.sort(key=lambda r: r['ann_roc_pct'], reverse=True)
    return results


def scan_ticker(symbol, filters, cached_info=None, max_retries=3):
    for attempt in range(max_retries + 1):
        try:
            return _fetch_and_screen(symbol, filters, cached_info), None
        except Exception as e:
            msg = str(e).lower()
            if ('rate' in msg or '429' in msg) and attempt < max_retries:
                backoff = DELAY * (2 ** attempt)
                print(f'    Rate limited on {symbol}, retry in {backoff:.0f}s...')
                time.sleep(backoff)
                continue
            return [], str(e)


# --- Global results store ---
scan_results_df = pd.DataFrame()


def on_scan_clicked(b):
    global scan_results_df
    selected = list(symbol_selector.value)
    if not selected:
        with scan_output:
            clear_output()
            print('Select at least one symbol from the list.')
        return

    filters = get_filters()
    all_results = []
    failed = []

    with scan_output:
        clear_output()
        print(f'Scanning {len(selected)} ticker(s): {", ".join(selected)}')
        print('-' * 50)

        info_map = {}
        if not preview_df.empty:
            info_map = preview_df.set_index('symbol').to_dict('index')

        for i, sym in enumerate(selected):
            print(f'[{i+1}/{len(selected)}] {sym}... ', end='')
            if i > 0:
                time.sleep(DELAY)
            cached = info_map.get(sym)
            results, error = scan_ticker(sym, filters, cached_info=cached)
            if error:
                print(f'FAILED: {error}')
                failed.append(sym)
            else:
                print(f'{len(results)} opportunities')
                all_results.extend(results)

        print('-' * 50)
        if not all_results:
            print('No opportunities found.')
            scan_results_df = pd.DataFrame()
            return

        scan_results_df = pd.DataFrame(all_results).sort_values('ann_roc_pct', ascending=False)
        print(f'\n{len(scan_results_df)} total opportunities found.\n')

        # Update post-scan filter options
        syms = sorted(scan_results_df['symbol'].unique().tolist())
        filter_sym_w.options = syms
        filter_sym_w.value = tuple(syms)

        display(_styled_table(scan_results_df))


def on_refine_clicked(b):
    if scan_results_df.empty:
        with refine_output:
            clear_output()
            print('No scan results. Run a scan first.')
        return

    df = scan_results_df.copy()

    # Filter by symbol
    selected_syms = filter_sym_w.value
    if selected_syms:
        df = df[df['symbol'].isin(selected_syms)]

    # Filter by rating
    selected_ratings = filter_rating_w.value
    if selected_ratings:
        df = df[df['rating'].isin(selected_ratings)]

    # Filter strong fundamentals
    if filter_strong_w.value:
        df = df[df['strong_fundamentals'] == True]

    # Sort
    col = sort_by_w.value
    ascending = sort_order_w.value
    df = df.sort_values(col, ascending=ascending)

    with refine_output:
        clear_output()
        if df.empty:
            print('No results match the selected filters.')
            return
        print(f'Showing {len(df)} of {len(scan_results_df)} results (sorted by {sort_by_w.label}, {"asc" if ascending else "desc"}):\n')
        display(_styled_table(df))


scan_btn.on_click(on_scan_clicked)
refine_btn.on_click(on_refine_clicked)

display(widgets.VBox([
    widgets.HBox([
        widgets.VBox([widgets.HTML('<b>Select Symbols</b>'), symbol_selector, select_hint]),
        filter_panel,
    ]),
    widgets.HTML('<br>'),
    scan_btn,
    scan_output,
    widgets.HTML('<hr>'),
    refine_panel,
]))

## 4. Visualizations

Run after a deep scan to see charts.

In [ ]:
if scan_results_df.empty:
    print('No scan results yet. Run a deep scan first.')
else:
    df = scan_results_df

    fig1 = px.scatter(
        df, x='delta', y='ann_roc_pct', color='rating_label', size='bid',
        hover_data=['symbol', 'strike', 'expiry', 'otm_pct'],
        title='ROC vs Delta by Rating',
        labels={'ann_roc_pct': 'Annualized ROC %', 'delta': 'Delta'},
    )
    fig1.show()

    fig2 = px.histogram(df, x='iv', nbins=20, title='Implied Volatility Distribution')
    fig2.show()

    top = df.nlargest(15, 'ann_roc_pct').copy()
    top['label'] = top['symbol'] + ' ' + top['strike'].apply(lambda x: f'${x:.0f}')
    fig3 = px.bar(top, x='label', y='ann_roc_pct', color='rating',
                  title='Top 15 by ROC', labels={'ann_roc_pct': 'ROC %'})
    fig3.show()

    fig4 = px.pie(df, names='dte', title='DTE Distribution')
    fig4.show()

## 5. Export

In [ ]:
if scan_results_df.empty:
    print('No results to export.')
else:
    filename = f'screener_results_{date.today().isoformat()}.csv'
    scan_results_df.to_csv(filename, index=False)
    print(f'Exported {len(scan_results_df)} rows to {filename}')